# Le climat des news pétrole (GDELT) comme régime de marché

**Ce qu'on fait, en une phrase :** on mesure l'humeur des médias sur le pétrole la veille de chaque rapport EIA, et on regarde si elle change la façon dont le prix réagit au rapport.

**Pourquoi GDELT :** c'est la seule source gratuite d'articles de presse sur le pétrole, avec un score de sentiment déjà calculé, sur 10 ans. GDELT lit les médias du monde entier depuis 2015, classe chaque article par thèmes (`ECON_OILPRICE` = prix du pétrole, `ENV_OIL` = pétrole en général) et lui donne un **ton** : la part de mots positifs moins la part de mots négatifs. Il faut un compte Google gratuit (sans carte bancaire) pour l'interroger via BigQuery.

**La news et le régime :**
- **La news**, c'est la surprise du rapport EIA du mercredi à 10h30, heure de New York (voir `news_eia.ipynb`). C'est le seul choc gratuit daté à la minute.
- **Le régime**, c'est le climat de news de la veille (mardi) : son **ton** (inquiétude ou optimisme des médias) et son **volume** (l'attention portée au pétrole).

**Tests fixés à l'avance, avant de regarder les résultats :**
1. Le climat de news change-t-il la réaction au rapport ? Hypothèses : plus d'attention → réaction plus forte ; ton plus négatif (stress) → réaction plus forte.
2. Le climat de news améliore-t-il la prédiction hors échantillon ?
3. Améliore-t-il la stratégie de 10h35 à 10h45 ?

**Ce qu'il faut dans `data/`** : `news_eia.csv` (créé par `news_eia.ipynb`), `eia_intraday.csv`, `eia_intraday_recent.csv`, `dataset.csv` et `wti_hmm_features.csv`.

## Étape 1 — Récupérer les news GDELT (une seule fois)

**Pourquoi seulement les mardis ?**
- **Pas d'information future :** un article du mercredi peut parler du rapport EIA lui-même. Le mardi, en heure UTC, se termine à 20h à New York, bien avant le rapport du mercredi 10h30.
- **Le quota gratuit :** BigQuery facture ce qu'il **lit**, pas ce qu'il renvoie, soit environ 1 Go par jour de GDELT. 600 mardis font environ 600 Go, ce qui tient dans le quota gratuit de 1 To par mois. 10 ans de jours complets n'y tiendraient pas.

**Marche à suivre (environ 10 min) :**
1. Lance la cellule ci-dessous : elle écrit la requête dans `data/gdelt_query.sql`.
2. Ouvre https://console.cloud.google.com/bigquery avec un compte Google (le « sandbox » gratuit suffit) et colle la requête.
3. **Avant de lancer**, regarde en haut à droite « This query will process … ». Si c'est plus de 1 To, raccourcis la période dans la cellule (par exemple `"2017-01-03"` au lieu de `"2015-03-03"`).
4. Lance, puis *Save results* → *CSV* → enregistre sous `data/gdelt_oil.csv`, et commit-le. Les autres n'auront pas besoin de compte.

In [10]:
import os, warnings
warnings.filterwarnings("ignore")
os.environ["LOKY_MAX_CPU_COUNT"] = "4"
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# Les mardis qui précèdent chaque rapport EIA, de mars 2015 (début de GDELT 2.0) à aujourd'hui
mardis = pd.date_range("2015-03-03", pd.Timestamp.today(), freq="W-TUE")
liste = ",\n    ".join(f"TIMESTAMP('{d:%Y-%m-%d}')" for d in mardis)
SQL = f"""SELECT
  DATE(PARSE_TIMESTAMP('%Y%m%d%H%M%S', CAST(DATE AS STRING))) AS jour,
  COUNT(*) AS n_articles,
  AVG(CAST(SPLIT(V2Tone, ',')[OFFSET(0)] AS FLOAT64)) AS ton
FROM `gdelt-bq.gdeltv2.gkg_partitioned`
WHERE _PARTITIONTIME IN (
    {liste})
  AND (V2Themes LIKE '%ECON_OILPRICE%' OR V2Themes LIKE '%ENV_OIL%')
GROUP BY jour
ORDER BY jour"""
os.makedirs("data", exist_ok=True)
open("data/gdelt_query.sql", "w").write(SQL)
print(len(mardis), "mardis -> requête écrite dans data/gdelt_query.sql")

609 mardis -> requête écrite dans data/gdelt_query.sql


In [15]:
!pip install pandas-gbq

   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ----------------------------------- ---- 3.4/3.8 MB 22.5 MB/s eta 0:00:01
   ----------------------------------- ---- 3.4/3.8 MB 22.5 MB/s eta 0:00:01
   ----------------------------------- ---- 3.4/3.8 MB 22.5 MB/s eta 0:00:01
   ----------------------------------- ---- 3.4/3.8 MB 22.5 MB/s eta 0:00:01
   ----------------------------------- ---- 3.4/3.8 MB 22.5 MB/s eta 0:00:01
   ---------------------------------------- 3.8/3.8 MB 2.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/5.6 MB ? eta -:--:--
   -------------------------- ------------- 3.7/5.6 MB 21.8 MB/s eta 0:00:01
   -------------------------- ------------- 3.7/5.6 MB 21.8 MB/s eta 0:00:01
   -------------------------- ------------- 3.7/5.6 MB 21.8 MB/s eta 0:00:01
   -------------------------- ------------- 3.7/5.6 MB 21.8 MB/s eta 0:00:01
   -------------------------- ------------- 3.7/5.6 MB 21.8 MB/s eta 0:00:01
   -------------


[notice] A new release of pip is available: 24.3.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
# À lancer UNE SEULE FOIS : récupère les news GDELT via BigQuery -> data/gdelt_oil.csv
# %pip install pandas-gbq          <- enlève le # la première fois pour installer, puis remets-le
import pydata_google_auth
from google.cloud import bigquery

PROJET = "eco-rune-513013-p4"       # identifiant affiché en haut de https://console.cloud.google.com/bigquery

if not os.path.exists("data/gdelt_oil.csv"):
    creds = pydata_google_auth.get_user_credentials(["https://www.googleapis.com/auth/bigquery"])   # fenêtre de connexion Google
    client = bigquery.Client(project=PROJET, credentials=creds)
    SQL = open("data/gdelt_query.sql").read()
    go = client.query(SQL, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)).total_bytes_processed / 1e9
    print(f"La requête va lire {go:.0f} Go (quota gratuit : ~1000 Go par mois)")
    if go < 950:
        client.query(SQL).to_dataframe().to_csv("data/gdelt_oil.csv", index=False)
        print("OK : data/gdelt_oil.csv créé")
    else:
        print("Trop gros, je ne lance pas : raccourcis la période dans la première cellule (ex. '2017-01-03').")
else:
    print("data/gdelt_oil.csv existe déjà : rien à faire")

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=262006177488-3425ks60hkk80fssi9vpohv88g6q1iqd.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fbigquery&state=rI3AVPi7R4KbGMTMXdGIJVzFGaElYu&code_challenge=5Tn9qDMt1tkRcMANbwbexQKZD0ETj6TfSMDcj8TU1-w&code_challenge_method=S256&prompt=consent&access_type=offline
La requête va lire 341 Go (quota gratuit : ~1000 Go par mois)
OK : data/gdelt_oil.csv créé


## Étape 2 — Le « climat de news » de la veille

Pour chaque mardi, on calcule deux variables :
- **`ton_z`** : le ton moyen des articles pétrole, comparé aux 2 années précédentes. Négatif = des médias plus inquiets que d'habitude.
- **`attention_z`** : le nombre d'articles pétrole, en log, comparé aux 2 années précédentes. Positif = plus de bruit médiatique que d'habitude.

On compare aux 2 années précédentes **seulement** : le volume de GDELT augmente avec les années, et il ne faut jamais utiliser le futur.

⚠️ **Le ton n'est pas une direction.** Une guerre au Moyen-Orient donne un ton très négatif, alors qu'elle fait **monter** le pétrole. On utilise donc le ton comme un niveau de stress, pas comme un signal d'achat ou de vente.

In [17]:
gd = pd.read_csv("data/gdelt_oil.csv", parse_dates=["jour"], index_col="jour").sort_index()
gd = gd[gd.index.dayofweek == 1]                       # on ne garde que les mardis : aucun article publié après le rapport

z = lambda s: (s - s.rolling(104, min_periods=52).mean().shift(1)) / s.rolling(104, min_periods=52).std().shift(1)
gdelt = pd.DataFrame({"ton_z": z(gd["ton"]),                         # ton des news pétrole, comparé aux 2 années précédentes
                      "attention_z": z(np.log(gd["n_articles"]))})  # volume de news pétrole, comparé aux 2 années précédentes
gdelt.index = gdelt.index + pd.Timedelta(days=1)       # rattaché au mercredi du rapport EIA
gdelt = gdelt.dropna()
gdelt.round(2).tail()

,ton_z,attention_z
jour,,
2026-09-30,-0.45,0.19
2026-10-07,-0.20,0.09
2026-10-14,-1.13,0.62
2026-10-21,-0.31,0.25
2026-10-28,0.73,-0.12


## Étape 3 — Assembler avec les rapports EIA et les autres régimes
`brut` = surprise sur les stocks de brut, en écarts-types, avec + = haussier. `B` = marché tendu la veille. Les probabilités du HMM sont calculées sans information future, comme dans `test_2`.

In [18]:
news = pd.read_csv("data/news_eia.csv", index_col=0, parse_dates=True)[["brut"]]     # surprise sur les stocks de brut (+ = haussier)
intra = pd.concat([pd.read_csv(f, index_col=0, parse_dates=True)
                   for f in ["data/eia_intraday.csv", "data/eia_intraday_recent.csv"] if os.path.exists(f)])
df = pd.read_csv("data/dataset.csv", index_col=0, parse_dates=True)

ev = (intra.join(news, how="inner").join(gdelt, how="inner")
           .join((df["slope"].shift(1) > 0).astype(int).rename("B")))           # B : marché tendu la veille (jusqu'en avril 2024)
ev["y"] = ev["r15"] - ev["r5"]                                                  # rendement tradable 10h35 -> 10h45
print(len(ev), "rapports EIA avec le climat GDELT de la veille, du", ev.index[0].date(), "au", ev.index[-1].date())

471 rapports EIA avec le climat GDELT de la veille, du 2016-03-02 au 2026-10-21


In [20]:
from hmmlearn.hmm import GaussianHMM
from scipy.stats import multivariate_normal
from sklearn.preprocessing import StandardScaler

feat = pd.read_csv("data/wti_hmm_features.csv", index_col=0, parse_dates=True).sort_index()
sc = StandardScaler().fit(feat[:"2018"])
X = sc.transform(feat)
hmm = GaussianHMM(3, covariance_type="full", n_iter=500, random_state=42).fit(X[feat.index <= "2018-12-31"])
ll = np.column_stack([multivariate_normal(hmm.means_[k], hmm.covars_[k]).logpdf(X) for k in range(3)])
a, p = np.zeros_like(ll), hmm.startprob_
for t in range(len(X)):                                  # filtre de Hamilton : P(régime | infos jusqu'à t)
    w = p * np.exp(ll[t] - ll[t].max())
    a[t] = w / w.sum()
    p = a[t] @ hmm.transmat_
order = np.argsort(sc.inverse_transform(hmm.means_)[:, 1])          # 0 calme, 1 normal, 2 crise
P = pd.DataFrame(a[:, order], index=feat.index, columns=["P0", "P1", "P2"]).shift(1)
ev = ev.drop(columns=["P0", "P1", "P2"], errors="ignore").join(P.reindex(ev.index, method="ffill"))

## Test 1 — Le climat de news change-t-il la réaction au rapport ?

Rendement de 10h29 à 10h34 sur la surprise, et sur la surprise multipliée par chaque régime. Avec + = haussier, le coefficient de `brut` doit être **positif**.
- **`brut:attention_z` > 0 et significatif** : quand on parle beaucoup de pétrole, le marché réagit plus fort au rapport.
- **`brut:ton_z` < 0 et significatif** : quand les médias sont inquiets, le marché réagit plus fort.
- La colonne « tous ensemble » est une course entre les trois régimes : celui qui reste significatif est celui qui compte.

In [21]:
def table(specs, data):
    out = {}
    for k, f in specs.items():
        m = smf.ols(f, data=data).fit(cov_type="HAC", cov_kwds={"maxlags": 4})
        out[k] = pd.concat([m.params, m.pvalues], keys=["coef", "p"])
    t = pd.DataFrame(out).round(3)
    return t.loc[[i for i in t.index if "brut" in i[1]]]          # on n'affiche que la réaction à la news et ses interactions

d = ev.loc[:"2024-04-05"].dropna(subset=["B", "P1"])               # période où les 3 régimes existent
print("Jusqu'en avril 2024 (les trois régimes existent) :")
print(table({"climat GDELT": "r5 ~ brut * (ton_z + attention_z)", "courbe (B)": "r5 ~ brut * B",
             "HMM": "r5 ~ brut * (P1 + P2)", "tous ensemble": "r5 ~ brut * (ton_z + attention_z + B + P1 + P2)"}, d))
print("\nToute la période GDELT (jusqu'à aujourd'hui) :")
table({"climat GDELT": "r5 ~ brut * (ton_z + attention_z)"}, ev)

Jusqu'en avril 2024 (les trois régimes existent) :
                       climat GDELT  courbe (B)    HMM  tous ensemble
coef brut                     0.148       0.194  0.211          0.252
     brut:B                     NaN      -0.101    NaN         -0.109
     brut:P1                    NaN         NaN -0.056         -0.045
     brut:P2                    NaN         NaN -0.109         -0.090
     brut:attention_z        -0.007         NaN    NaN          0.017
     brut:ton_z               0.014         NaN    NaN          0.017
p    brut                     0.000       0.000  0.000          0.000
     brut:B                     NaN       0.040    NaN          0.044
     brut:P1                    NaN         NaN  0.390          0.519
     brut:P2                    NaN         NaN  0.126          0.251
     brut:attention_z         0.781         NaN    NaN          0.540
     brut:ton_z               0.454         NaN    NaN          0.476

Toute la période GDELT (jusqu'à aujour

climat GDELT
coef brut                     0.132
     brut:ton_z               0.016
     brut:attention_z        -0.017
p    brut                     0.000
     brut:ton_z               0.346
     brut:attention_z         0.356

## Test 2 — Prédiction hors échantillon
Chaque année depuis 2019 est prédite avec un modèle estimé sur les 3 années précédentes. Un R² positif veut dire mieux que la moyenne. Le climat de news est utile si la deuxième ligne fait mieux que la première.

In [22]:
def rolling(f, data, annees, cible):
    """Chaque année est prédite par un modèle estimé sur les 3 années précédentes (aucune information future)."""
    o = []
    for yr in annees:
        tr, te = data.loc[str(yr - 3):str(yr - 1)], data.loc[str(yr)]
        o.append(pd.DataFrame({"y": te[cible], "pred": smf.ols(f, data=tr).fit().predict(te), "bench": tr[cible].mean()}))
    return pd.concat(o)

ANNEES = range(2019, pd.Timestamp.today().year + 1)
for f in ["r5 ~ brut", "r5 ~ brut * (ton_z + attention_z)"]:
    o = rolling(f, ev, ANNEES, "r5")
    print(f"{f:36s} R2 hors échantillon = {1 - ((o.y - o.pred) ** 2).sum() / ((o.y - o.bench) ** 2).sum():.4f}")

r5 ~ brut                            R2 hors échantillon = 0.0347
r5 ~ brut * (ton_z + attention_z)    R2 hors échantillon = -0.0397


## Test 3 — Stratégie de 10h35 à 10h45
Mêmes règles que dans `test_2` : achat si la dérive prédite est positive, vente sinon, coût de 0,04 %. On sépare la période récente, jamais utilisée pour la stratégie. Une t-stat d'écart supérieure à 2 veut dire que le climat de news améliore significativement la stratégie.

In [23]:
COST = 0.04
def perf(p, ref=None):
    ans = (p.index[-1] - p.index[0]).days / 365.25
    r = {"trades": len(p), "Sharpe net": p.mean() / p.std() * np.sqrt(len(p) / ans), "t-stat": p.mean() / p.std() * np.sqrt(len(p))}
    if ref is not None:
        diff = p - ref
        r["t-stat écart vs base"] = diff.mean() / diff.std() * np.sqrt(len(diff))
    return r

pnl = {}
for k, f in {"base (brut seul)": "y ~ brut", "+ climat GDELT": "y ~ brut * (ton_z + attention_z)"}.items():
    o = rolling(f, ev, ANNEES, "y")
    pnl[k] = np.sign(o.pred) * o.y - COST                   # achat si dérive prédite > 0, vente sinon
res = {}
for per, deb in {"2019 -> aujourd'hui": "2019", "dont avril 2024 -> aujourd'hui": "2024-04-06"}.items():
    for k, p in pnl.items():
        res[(per, k)] = perf(p.loc[deb:], None if k.startswith("base") else pnl["base (brut seul)"].loc[deb:])
pd.DataFrame(res).T.round(2)

trades  Sharpe net  t-stat  \
2019 -> aujourd'hui            base (brut seul)   346.0        0.16    0.45   
                               + climat GDELT     346.0        0.08    0.21   
dont avril 2024 -> aujourd'hui base (brut seul)   112.0       -0.90   -1.42   
                               + climat GDELT     112.0       -1.30   -2.07   

                                                 t-stat écart vs base  
2019 -> aujourd'hui            base (brut seul)                   NaN  
                               + climat GDELT                   -0.41  
dont avril 2024 -> aujourd'hui base (brut seul)                   NaN  
                               + climat GDELT                   -0.50

## Est-ce que ça suit le README ?

| Ce que prévoit le README | Ce qu'on fait | Conforme ? |
|---|---|---|
| **1. Régimes latents (HMM-TVTP)** | HMM à 3 états du collègue, probabilités de la veille, sans information future | Oui pour le HMM. Pas de TVTP : les transitions restent constantes. |
| … transitions qui dépendent du *sentiment agrégé des news passées* | Le climat GDELT de la veille sert directement de variable de régime | En partie : même idée, en version simplifiée (variable observée au lieu d'entrer dans les transitions du HMM) |
| **2. Chocs de news extraits par NLP/LLM** | Le choc est la surprise du rapport EIA (chiffre officiel, par type de choc dans `news_eia`). Le ton GDELT est un score NLP déjà calculé (par dictionnaire). | En partie : les types de choc oui, le LLM non (pas de flux d'articles datés à la minute gratuit) |
| **3. Mélange d'experts conditionné au régime** | Régression `brut × régime` : un expert linéaire par régime, pondéré par le régime | Oui (experts linéaires) |
| **Principe : régime estimé avant la news** | Climat du mardi, HMM et courbe de la veille | Oui |
| **Ablation** | News seule, + courbe (régime dur), + HMM (régime souple), + climat GDELT | Oui |
| **Hors échantillon, coûts, Sharpe** | Fenêtre glissante de 3 ans, coût de 0,04 %, période récente séparée | Oui |

**En résumé :** l'architecture et la méthode suivent le README. Les deux écarts, c'est l'absence de LLM et de TVTP. Ils sont dus aux données gratuites disponibles, et c'est à écrire tel quel dans les limites.